# Current ML Provider Profiling

This notebook reads the current four-step validation runs, rather than using the older hard-coded 1920x1080 benchmark data. It combines timing_and_parameters.txt with Score-P call-tree regions.

Current validation jobs: No-ML 2299999, PhyDLL 2303466, AIX 2303872, SmartSim 2349847.

In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_ROOT = Path('/hpcwork/thes2181/mini_app')
if not OUTPUT_ROOT.exists():
    OUTPUT_ROOT = Path('/rwthfs/rz/cluster/hpcwork/ro092286/smartsim/mini_app')
PROFILE_ROOT = Path('/rwthfs/rz/cluster/hpcwork/ro092286/smartsim/mini_app/scorep_runs')
JOB_IDS = {'No-ML': 2299999, 'PhyDLL': 2303466, 'AIX': 2303872, 'SmartSim': 2349847}
print('Timing root:', OUTPUT_ROOT)
print('Score-P root:', PROFILE_ROOT)

In [ ]:
def block_value(text, label):
    match = re.search(rf'^\s*{re.escape(label)}:\s*(.+)$', text, re.MULTILINE)
    return match.group(1).strip() if match else None

def number(text, label):
    value = block_value(text, label)
    return float(value) if value is not None else None

def gib(text, label):
    match = re.search(rf'^\s*{re.escape(label)}:\s*([\d.]+)\s*GiB', text, re.MULTILINE)
    return float(match.group(1)) if match else None

def find_timing_file(job_id):
    for path in OUTPUT_ROOT.glob('**/timing_and_parameters.txt'):
        text = path.read_text(errors='replace')
        if re.search(rf'^Slurm Job ID: {job_id}$', text, re.MULTILINE):
            return path
    return None

def parse_timing(provider, job_id):
    path = find_timing_file(job_id)
    if path is None:
        return {'provider': provider, 'job_id': job_id, 'path': None}
    text = path.read_text(errors='replace')
    record = {'provider': provider, 'job_id': job_id, 'path': str(path)}
    for key in ['TARGET_WIDTH', 'TARGET_HEIGHT', 'MPI_RANKS', 'RANK_GRID_X', 'RANK_GRID_Z']:
        record[key.lower()] = block_value(text, key)
    for key in ['Regular', 'ML']:
        match = re.search(rf'^\s*{key}: count=(\d+), total_ms=([\d.]+), avg_ms=([\d.]+), min_ms=([\d.]+), max_ms=([\d.]+)$', text, re.MULTILINE)
        if match:
            record[f'{key.lower()}_count'] = int(match.group(1))
            record[f'{key.lower()}_total_ms'] = float(match.group(2))
            record[f'{key.lower()}_avg_ms'] = float(match.group(3))
            record[f'{key.lower()}_min_ms'] = float(match.group(4))
            record[f'{key.lower()}_max_ms'] = float(match.group(5))
    for key in ['prepare_data', 'put_tensor', 'run_model', 'unpack', 'ml_total_wall', 'ml_accounted']:
        match = re.search(rf'^\s*{key}: count=(\d+), total_s=([\d.]+), avg_s=([\d.]+)$', text, re.MULTILINE)
        if match:
            record[f'{key}_total_s'] = float(match.group(2))
            record[f'{key}_avg_s'] = float(match.group(3))
    for key in ['Preparation', 'Compilation', 'Solving', 'Total']:
        record[f'{key.lower()}_s'] = number(text, key)
    record['traffic_input_gib'] = gib(text, 'input')
    record['traffic_output_gib'] = gib(text, 'output')
    return record

timings = pd.DataFrame([parse_timing(provider, job_id) for provider, job_id in JOB_IDS.items()])
timings

## Aggregate step timings
The current timing files contain regular/ML aggregate timings. SmartSim additionally records prepare, put, model, and unpack timings. PhyDLL and AIX currently expose those details through Score-P regions instead of the text parser.

In [ ]:
plot = timings.set_index('provider')[['regular_avg_ms', 'ml_min_ms', 'ml_max_ms']].rename(columns={'regular_avg_ms': 'Regular', 'ml_min_ms': 'ML minimum', 'ml_max_ms': 'ML maximum'})
plot.plot(kind='bar', figsize=(10, 5), ylabel='Milliseconds', title='Current validation step timings')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
substep_columns = ['prepare_data_avg_s', 'put_tensor_avg_s', 'run_model_avg_s', 'unpack_avg_s']
available = timings.set_index('provider')[substep_columns].dropna(how='all')
if not available.empty:
    available.rename(columns=lambda x: x.replace('_avg_s', ''), inplace=True)
    available.plot(kind='bar', stacked=True, figsize=(10, 5), ylabel='Seconds', title='Current ML substeps recorded in timing files')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print('No text-based substep timings found.')

## Score-P time breakdown

The CUBE export below uses Score-P `max_time`: the maximum inclusive wall time across ranks. The root of each hierarchy is the measured `solver_step_ml_steady` wall time. Nested measurements are absolute wall times and are intentionally not forced to add up to their parent.

Only the current C++ PhyDLL client is included. The older Python-client results in `benchmark_visualization.ipynb` are separate experiments and are deliberately not mixed into this comparison.

In [ ]:
cube_dump = shutil.which('cube_dump') or '/hpcwork/ro092286/smartsim/CPP-ML-Interface/tmp/opencode/scorep-8.4-papi72-install/bin/cube_dump'
rows = []
for provider, job_id in JOB_IDS.items():
    profile = PROFILE_ROOT / f'terrain_solver_coupled_{job_id}_rank_0' / 'profile.cubex'
    if profile.exists() and Path(cube_dump).exists():
        result = subprocess.run([cube_dump, '-m', 'max_time', '-c', 'all', '-t', 'aggr', '-s', 'human', '-o', '-', str(profile)], capture_output=True, text=True, check=False)
        for line in result.stdout.splitlines():
            match = re.match(r'^(.+?)\(id=(\d+)\)\s+([0-9.eE+-]+)$', line.strip())
            if match:
                rows.append({'provider': provider, 'job_id': job_id, 'region': match.group(1).strip(), 'region_id': int(match.group(2)), 'wall_s': float(match.group(3))})
scorep = pd.DataFrame(rows)
scorep.head()

In [ ]:
shared_regions = ['solver_ml_prepare_input', 'solver_ml_provider_call', 'solver_ml_output_copy']
shared = scorep.sort_values('region_id').drop_duplicates(['provider', 'region'], keep='last').query('region in @shared_regions').pivot(index='provider', columns='region', values='wall_s').fillna(0)
shared = shared.reindex(columns=shared_regions, fill_value=0).rename(columns=lambda x: x.replace('solver_ml_', ''))
shared.plot(kind='bar', figsize=(10, 5), ylabel='Maximum wall seconds', title='Shared solver-side ML stages')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
shared

## Provider-specific absolute stages

A sunburst or treemap is not valid for these measurements. Score-P `max_time` is an inclusive maximum independently selected across ranks; nested regions overlap, and their maxima may come from different ranks. The per-provider charts below show absolute stage values against the complete steady ML-step maximum without adding or partitioning them.

In [ ]:
# Category mapping and leaf lookup
leaf_hierarchy = {
    'PhyDLL': {
        'Solver-side protocol': ['phydll_prepack', 'phydll_send', 'phydll_recv', 'phydll_unpack'],
        'DL-client receive and reshape': ['dl_recv', 'dl_input_unpack'],
        'DL-client accelerator': ['dl_h2d', 'dl_torch_forward', 'dl_d2h'],
        'DL-client output': ['dl_output_reorder', 'dl_send'],
    },
    'AIX': {
        'Service data movement': ['gatherInputData', 'scatterOutputData'],
        'Host-side work': ['inferenceHost'],
        'Accelerator': ['h2d_copy', 'torchInference::forward', 'd2h_copy'],
    },
    'SmartSim': {
        'Client preparation': ['smartsim_prepare_input'],
        'SmartRedis transport': ['smartsim_put_tensor', 'smartsim_unpack_tensor'],
        'RedisAI execution': ['smartsim_run_model'],
    },
}

def steady_root(provider):
    row = scorep[(scorep.provider == provider) & (scorep.region == 'solver_step_ml_steady')].sort_values('region_id').tail(1)
    return row.iloc[0].wall_s if not row.empty else 0.0

def hierarchy_leaves(provider):
    values = scorep[scorep.provider == provider].sort_values('region_id').drop_duplicates('region', keep='last').set_index('region').wall_s
    rows = []
    for category, leaves in leaf_hierarchy[provider].items():
        for leaf in leaves:
            value = values.get(leaf, 0.0)
            if value > 0:
                rows.append({'provider': provider, 'category': category, 'region': leaf, 'wall_s': value})
    return pd.DataFrame(rows)

# Inclusive maximum wall times are not additive. A bar may extend past the
# complete ML-step line because the maxima can be from different ranks.
category_colors = dict(zip(
    sorted({category for mapping in leaf_hierarchy.values() for category in mapping}),
    plt.get_cmap('tab10').colors,
))

def plot_provider_stages(provider):
    stages = hierarchy_leaves(provider).sort_values('wall_s')
    root = steady_root(provider)
    if stages.empty:
        print(f'No stage data for {provider}.')
        return
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.48 * len(stages) + 1.5)))
    ax.axvspan(0, root, color='#e8eef5', zorder=0)
    ax.axvline(root, color='#1f4e79', linewidth=2, zorder=3)
    ax.barh(stages.region, stages.wall_s, color=[category_colors[c] for c in stages.category], zorder=2)
    for y, value in enumerate(stages.wall_s):
        ax.text(value, y, f'  {value:.3f} s', va='center', fontsize=9)
    ax.set_xlim(0, max(root, stages.wall_s.max()) * 1.20)
    ax.set_xlabel('Score-P max_time: inclusive maximum wall seconds')
    ax.set_title(f'{provider}: measured steady-state stages (overlap is allowed)')
    categories = list(stages.category.drop_duplicates())
    handles = [plt.Rectangle((0, 0), 1, 1, color=category_colors[c]) for c in categories]
    handles += [plt.Line2D([0], [0], color='#1f4e79', linewidth=2)]
    ax.legend(handles, categories + [f'Complete ML step ({root:.3f} s)'], loc='lower right', fontsize=8)
    ax.grid(axis='x', alpha=0.25, zorder=0)
    plt.tight_layout()
    plt.show()

for provider in leaf_hierarchy:
    plot_provider_stages(provider)


## Instrumentation interpretation and next measurements

Do not derive coverage or an unaccounted residual by subtracting these values. `max_time` is inclusive, and each region independently reports the slowest rank. The AIX stage total exceeding the ML-step maximum is expected under that definition; it proves these values are not a partition. The small SmartSim stage values identify narrow regions worth expanding, but they do not prove that the remainder is missing time.

### Useful additions

1. Add contiguous, non-overlapping solver-side regions around the full provider path: preparation, invocation, result application, and synchronization. These can be summed on the same rank.
2. For SmartSim, split the current `smartsim_run_model` region around client-side request construction, submission, wait/synchronization, and response handling.
3. For AIX, add child regions around the host-side setup that precedes `gatherInputData` and the completion work after `scatterOutputData`; retain Score-P MPI callpaths rather than wrapping MPI calls redundantly.
4. For PhyDLL, create per-request regions in the DL-client process, with matching request identifiers or a separately profiled client CUBE. The current client-wide region crosses warmup and steady calls.
5. Record a per-rank summary for the same step and use one selected rank, or Score-P severity data by rank, whenever an additive breakdown is required.


In [ ]:
stage_rows = []
for provider in leaf_hierarchy:
    stage_rows.append({'provider': provider, 'category': 'Complete ML step', 'stage': 'solver_step_ml_steady', 'wall_s': steady_root(provider)})
    stage_rows.extend(hierarchy_leaves(provider).to_dict('records'))
stage_table = pd.DataFrame(stage_rows).sort_values(['provider', 'category', 'wall_s'], ascending=[True, True, False])
display(stage_table[['provider', 'category', 'stage', 'wall_s']])

provider_roots = stage_table[stage_table.stage == 'solver_step_ml_steady'].set_index('provider').wall_s.sort_values(ascending=False)
provider_roots.plot(kind='bar', figsize=(7, 4), ylabel='Steady ML maximum wall seconds', title='Whole ML-step time by provider')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
provider_roots


### Reading the charts
Each chart is one provider. The blue line and shaded band are the full `solver_step_ml_steady` maximum. Each coloured bar is an inclusive `max_time` measurement. Bars are deliberately not stacked and may pass the line: they overlap and can have maxima on different ranks.

Use the chart to identify large measured intervals for follow-up instrumentation, not to assign a percentage composition of the ML step.